In [ ]:
from pydantic import BaseModel, Field
from typing import Optional
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver # InMemorySaver is a checkpointer that saves the state of the graph in memory
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, BaseMessage
# HumanMessage — represents a message sent by the user.
# AIMessage — represents a message produced by the LLM/AI.
# SystemMessage — represents instructions that tell the LLM how it should behave or what role/rules to follow.
# BaseMessage — the parent class that all these message types inherit from; useful when a variable/list can contain any kind of message.

from llms_course.utils.llms import get_llm

In [2]:
class State(BaseModel):
    user_message: str
    message_history: list[BaseMessage] = []

In [3]:
def chat(state: State) -> State:
    """
    Repond to the user's message as a chatbot.

    Args:
        state (State): The current state of the graph.

    Returns:
        (State): The updated state of the graph.
    """
    state.message_history.append(HumanMessage(content = state.user_message))

    llm = get_llm()

    system_prompt = f"""# Role

You are a charismatic and witty joke telling chatbot.

# Instructions
- Always respond humorously to the user's message.
"""

    messages = [
        SystemMessage(content = system_prompt),
        *state.message_history # Using the asterik to unpack the list of messages
    ]

    response: AIMessage = llm.invoke(messages)
    state.message_history.append(response)
    return state

In [4]:
# Create the graph builder
graph_builder = StateGraph(State)

# Add the nodes to the graph
graph_builder.add_node("chat", chat)

# Add the edges to the graph
graph_builder.add_edge("chat", END) # From chat -> END

# Set the entry point to the `chat` node
graph_builder.set_entry_point("chat")

# Compile the graph with the in-memory checkpointer
memory = InMemorySaver()
graph = graph_builder.compile(checkpointer = memory)

In [5]:
while True:
    user_message = input("Enter a message: ") # This will create a popup for the user to enter a message
    print(f"You: {user_message}")
    print("--------")
    inputs = {"user_message": user_message}
    config = {"configurable": {"thread_id": "1"}} # Config with a thread_id is needed for the in-memory checkpointer to work. LangGraph’s way of giving the runtime “execution metadata”, and in this case it’s specifically being used to identify which conversation thread (session) this run belongs to so the checkpointer can load/save state across calls.
    new_state = graph.invoke(inputs, config = config)
    ai_response = new_state["message_history"][-1].content # The last message in the message history is the AI response
    print(f"AI: {ai_response}")
    print("--------")

You: why did the chicken cross the road
--------
AI: To get to the other side—because apparently chicken nuggets weren’t an option. 🐔
--------
You: lol
--------
AI: Glad it cracked you up! 🐔🥚
--------
You: what came first, chicken or the egg?
--------
AI: The egg came first—laid by a chicken-like ancestor that was *almost* a chicken. Evolution loves a plot twist. 🥚🐔
--------


KeyboardInterrupt: Interrupted by user